# Lab 02 · Instructor solution

Complete reference for [the learner notebook](../02-analysis-pass.ipynb). Try the exercise before opening this solution.

The pass publishes a **top-level structural inventory**. It is not a cost model, recursive count, or optimization. Learner understanding is assessed separately from passing public checks.

In [ ]:
import sys

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import Gate
from qiskit.converters import circuit_to_dag
from qiskit.transpiler import AnalysisPass, PassManager

from qclab.notebook import check_solution

print("Python:", sys.version.split()[0], "| Qiskit:", qiskit.__version__)
assert qiskit.__version__ == "2.6.0", "Select this project's .venv kernel."

In [ ]:
class CircuitReport(AnalysisPass):
    """Publish a top-level operation inventory without modifying the DAG."""

    def run(self, dag):
        counts = {}
        two_qubit_gates = 0
        for node in dag.op_nodes():
            counts[node.name] = counts.get(node.name, 0) + 1
            if isinstance(node.op, Gate) and len(node.qargs) == 2:
                two_qubit_gates += 1
        self.property_set["circuit_report"] = {
            "num_qubits": dag.num_qubits(),
            "num_clbits": dag.num_clbits(),
            "operation_counts": counts,
            "num_two_qubit_gates": two_qubit_gates,
        }

## Why this implementation works

- Read-only iteration leaves the DAG intact.
- Every operation occurrence contributes to the name histogram.
- Type plus arity distinguishes gates from two-wire instructions or barriers.
- New local counters and a new report prevent state leaking between runs.
- Only `circuit_report` is assigned, preserving other passes' information.
- No explicit return is needed: Python returns `None`.

Gate definitions and control-flow blocks are deliberately not expanded. A composite two-qubit gate contributes one; an `if_else` containing pair gates contributes zero to the pair-gate count.

In [ ]:
circuit = QuantumCircuit(3, 2, name="inventory_example")
circuit.h(0)
circuit.h(1)
circuit.cx(0, 1)
circuit.barrier(0, 1)
circuit.measure(1, 0)
circuit.reset(1)
circuit.delay(10, 0)
print(circuit.draw(output="text"))

In [ ]:
manager = PassManager(CircuitReport())
output = manager.run(circuit)
print(manager.property_set["circuit_report"])
assert output == circuit
assert manager.property_set["circuit_report"]["num_two_qubit_gates"] == 1

In [ ]:
result = check_solution(CircuitReport, lab="02")
print(result.output)
print("Acceptance status:", "PASS" if result.passed else "NOT YET")
assert result.passed, result.output

## Example regression: a barrier is not a gate

Two qubit arguments alone are insufficient. This test catches an implementation that uses only arity. The empty second run also exposes accumulated counts.

In [ ]:
boundary = QuantumCircuit(2)
boundary.barrier(0, 1)
boundary.cx(0, 1)
analysis = CircuitReport()
analysis.run(circuit_to_dag(boundary))
old_report = analysis.property_set["circuit_report"]
assert old_report["operation_counts"] == {"barrier": 1, "cx": 1}
assert old_report["num_two_qubit_gates"] == 1

analysis.run(circuit_to_dag(QuantumCircuit(2)))
assert analysis.property_set["circuit_report"]["operation_counts"] == {}
assert old_report["operation_counts"] == {"barrier": 1, "cx": 1}
print("Barrier and report-isolation regressions: PASS")
print("Gate base class:", Gate.__name__)

## Facilitation notes

Ask learners to predict results before running cells. If a learner copies this reference, passing tests demonstrates execution, not understanding. Ask them to explain a new example and write an independent regression.

Common mistakes: counting unique names, treating every two-wire node as a gate, hard-coding `cx`, returning a dictionary rather than publishing it, mutating the DAG, and retaining a shared mutable counter.

For control flow, ask why a top-level count cannot predict how often a branch executes. For composite gates, ask why decomposition changes the inventory even when the represented computation stays equivalent.

No general completion-time claim, learning gain, or maintainer endorsement has been established. Capture observations with the [facilitator guide](../../docs/facilitator-guide.md). Use the [learner contract](../02-analysis-pass.ipynb) as the authoritative behavior specification.